In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Complete 5-Class ESI Multinomial Logistic Regressor with Direct Inverse Class Weighting (`models/lr_multiclass.ipynb`)

This notebook trains a **Complete 5-Class Multinomial Logistic Regressor (ESI 1 to 5)** using **13 Clinical Feature Engineered Inputs** and **Direct Inverse Class Frequency Weighting**:

### System Architecture & Features
1. **Complete 5-Class Output**: Predicts ESI levels 1, 2, 3, 4, and 5 directly.
2. **Feature Engineering Inputs**: Uses 13 clinical feature engineered inputs (age, gender, breathing difficulty, dyspnea flags, vital sign anomaly flags).
3. **Direct Inverse Class Frequency Weighting**: Calculates exact inverse frequency weights for all 5 classes $w_{\text{inv}, c} = N_{\text{total}} / N_c$ and applies sample weights directly.
4. **Class Count Comparison Reports & CSV Exports**: Reports actual vs predicted counts, Precision, Recall, and PR-AUC. Exports `reports/lr_multiclass_val_report.csv` and `reports/lr_multiclass_test_report.csv`.
5. **Model Artifact Export**: Saved to `deploy/lr_multiclass_model.rds`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(nnet)
library(dplyr)
library(ggplot2)
library(tidyr)
library(pROC)

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

config <- fromJSON(config_path)

cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Construct 13 FE Inputs & Complete Case Filtering
# ---------------------------------------------------------
set.seed(config$training$random_state)

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

cat("Loading dataset from:", data_file, "...\n")

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))

raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0

# Construct 13 Clinical Feature Engineered Inputs
df_feng <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0)
)

raw_esi <- as.character(raw_df[[target_col]])
df_feng$raw_esi <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))

initial_rows <- nrow(df_feng)
df <- na.omit(df_feng)
cat(sprintf("Complete Case Filtering: Removed %d rows with NULL/NA features (Remaining complete rows: %d)\n",
            initial_rows - nrow(df), nrow(df)))

cat(sprintf("Dataset Ready: %d rows x %d cols\n", nrow(df), ncol(df)))
cat("Full 5-Class ESI Distribution:\n")
print(table(df$raw_esi))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Data Partitioning & Feature Scaling
# ---------------------------------------------------------
set.seed(config$training$random_state)

test_size <- config$training$test_size
val_size  <- config$training$val_size

# Stratified Test split (15%)
in_train_val <- createDataPartition(df$raw_esi, p = 1 - test_size, list = FALSE)
train_val_df <- df[in_train_val, ]
test_df      <- df[-in_train_val, ]

# Stratified Validation split (15%)
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$raw_esi, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]

# Standardize continuous feature (age)
cont_cols <- "age"
preproc <- preProcess(train_df[, cont_cols, drop = FALSE], method = c("center", "scale"))

train_df <- predict(preproc, train_df)
val_df   <- predict(preproc, val_df)
test_df  <- predict(preproc, test_df)

cat(sprintf("Partition sizes:\n  Train: %d rows\n  Val:   %d rows\n  Test:  %d rows\n",
            nrow(train_df), nrow(val_df), nrow(test_df)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Calculate Direct Inverse Class Frequency Weights & Train 5-Class Model (No Multipliers)
# ---------------------------------------------------------
set.seed(config$training$random_state)

n_total <- nrow(train_df)
class_counts <- table(train_df$raw_esi)
inv_weights  <- n_total / class_counts

cat("=== Direct 5-Class Inverse Class Frequency Weight Calculation ===\n")
for (cls in names(inv_weights)) {
  cat(sprintf("  - ESI Class %s: Count = %d / %d -> Direct Inverse Weight = %.4f\n",
              cls, class_counts[cls], n_total, inv_weights[cls]))
}
cat("\n")
sample_weights <- as.numeric(inv_weights[as.character(train_df$raw_esi)])
feat_names <- setdiff(names(train_df), c("raw_esi"))
formula_lr <- as.formula(paste("raw_esi ~", paste(feat_names, collapse = " + ")))
cat("Training 5-Class Multinomial Logistic Regressor with Direct Inverse Class Weights...\n")
lr_final <- multinom(formula_lr, data = train_df, weights = sample_weights, trace = FALSE, MaxNWts = 5000)
cat("5-Class Logistic Regression Training Complete!\n")
print(summary(lr_final))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Evaluate 5-Class Benchmark Across Splits & Export CSV Reports
# ---------------------------------------------------------
calc_pr_auc <- function(actual_binary, prob_positive) {
  tryCatch({
    ord <- order(prob_positive, decreasing = TRUE)
    act_sorted <- (actual_binary[ord] == 1)
    tp <- cumsum(act_sorted)
    fp <- cumsum(!act_sorted)
    n_pos <- sum(act_sorted)
    if (n_pos == 0) return(NA)
    rec <- c(0, tp / n_pos)
    prec <- c(tp[1] / max(1, tp[1] + fp[1]), tp / (tp + fp))
    dx <- diff(rec)
    my <- (prec[-1] + prec[-length(prec)]) / 2
    return(as.numeric(sum(dx * my)))
  }, error = function(e) NA)
}
evaluate_multiclass_lr <- function(model, data, set_name) {
  prob_matrix <- predict(model, newdata = data, type = "probs")
  target_classes <- c("1", "2", "3", "4", "5")
  
  max_idx <- max.col(prob_matrix, ties.method = "first")
  pred_factor <- factor(colnames(prob_matrix)[max_idx], levels = target_classes)
  actual_factor <- factor(data$raw_esi, levels = target_classes)
  
  cm <- confusionMatrix(pred_factor, actual_factor)
  acc <- as.numeric(cm$overall["Accuracy"])
  
  prec_by_class <- cm$byClass[, "Pos Pred Value"]
  rec_by_class  <- cm$byClass[, "Sensitivity"]
  macro_prec    <- mean(prec_by_class, na.rm = TRUE)
  macro_rec     <- mean(rec_by_class,  na.rm = TRUE)
  
  pr_auc_by_class <- numeric(5)
  names(pr_auc_by_class) <- target_classes
  for (cls in target_classes) {
    act_bin <- ifelse(actual_factor == cls, 1, 0)
    pr_auc_by_class[cls] <- calc_pr_auc(act_bin, prob_matrix[, cls])
  }
  macro_pr_auc <- mean(pr_auc_by_class, na.rm = TRUE)
  
  roc_obj <- pROC::multiclass.roc(actual_factor, prob_matrix)
  macro_roc_auc <- as.numeric(roc_obj$auc)
  
  actual_table <- table(actual_factor)
  pred_table   <- table(pred_factor)
  
  diff_vec <- as.numeric(pred_table) - as.numeric(actual_table)
  diff_str <- ifelse(diff_vec >= 0, paste0("+", diff_vec), as.character(diff_vec))
  
  class_comparison <- data.frame(
    Class        = target_classes,
    Actual_Count = as.numeric(actual_table),
    Pred_Count   = as.numeric(pred_table),
    Diff         = diff_str,
    Precision    = round(prec_by_class, 4),
    Recall       = round(rec_by_class, 4),
    PR_AUC       = round(pr_auc_by_class, 4)
  )
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   DIRECT INVERSE CLASS-WEIGHTED 5-CLASS LR - %s SET BENCHMARK\n", toupper(set_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Overall Accuracy        : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  Macro Precision         : %.4f (%.2f%%)\n", macro_prec, macro_prec * 100))
  cat(sprintf("  Macro Recall (Sens)     : %.4f (%.2f%%)\n", macro_rec, macro_rec * 100))
  cat(sprintf("  Macro PR-AUC            : %.4f\n", macro_pr_auc))
  cat(sprintf("  Multi-Class ROC-AUC     : %.4f\n", macro_roc_auc))
  cat(sprintf("============================================================\n\n"))
  
  cat("Target Class Count Comparison & Performance Summary Table:\n")
  print(class_comparison)
  
  cat("\nFull 5x5 Confusion Matrix (Rows: Predicted, Columns: Actual):\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
  return(class_comparison)
}
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
# Benchmark Model on Validation set & Export CSV
val_report <- evaluate_multiclass_lr(lr_final, val_df, "Validation")
write.csv(val_report, file = file.path(reports_dir, "lr_multiclass_val_report.csv"), row.names = FALSE)
cat("Validation Evaluation CSV Report written to: reports/lr_multiclass_val_report.csv\n")
# Benchmark Model on Test set & Export CSV
test_report <- evaluate_multiclass_lr(lr_final, test_df, "Test")
write.csv(test_report, file = file.path(reports_dir, "lr_multiclass_test_report.csv"), row.names = FALSE)
cat("Test Evaluation CSV Report written to: reports/lr_multiclass_test_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Save Optimal 5-Class Model Artifacts
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
model_path <- file.path(deploy_dir, "lr_multiclass_model.rds")
saveRDS(list(model = lr_final, preproc = preproc, inv_weights = inv_weights), file = model_path)
cat("Direct Inverse Class Weighted 5-Class Logistic Regressor saved to:", model_path, "\n")